# Predictive Anayltics: Support Vector Machines with Regression for Census Tract

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [ ]:
from run_config import PATHS

In [ ]:
#TODO: embedding ansehen -> vielleicht austauschen
#TODO: make more time efficient
#TODO: change to thundersvm -> installation umstädlich

In [ ]:
DO_GRID_SEARCH = True
GRID_SAMPLE = 2_000
SPATIAL_ENCODING = "latlong" # options: embedding, latlong, onehot
MODE = "full" # options: full, sample
TIME_UNIT = "4H" # options: 1H, 4H, 24H
H3_RES = "7" # options 7,8

In [22]:
CENSUS_PATH = "../data/full/raw_data/Census_Tracts.csv"
COMM_PATH = "../data/full/raw_data/Community_Areas.csv"

In [23]:
import pandas as pd
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVR 
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import Nystroem
from joblib import load, dump
import h3

from shapely import wkt

import geopandas as gpd
from shapely.geometry import Polygon
from srai.neighbourhoods import H3Neighbourhood
from srai.loaders import OSMOnlineLoader
from srai.joiners import IntersectionJoiner
from srai.h3 import ring_buffer_h3_regions_gdf
from srai.embedders import Hex2VecEmbedder

import networkx as nx
from libpysal.weights import Queen
from node2vec import Node2Vec

## Preparations

In [24]:
INPUT = "../data/" + MODE + "/train_test_data/" 

In [ ]:
SPATIAL_UNIT = "CENSUS_TRACTS"

# Paths, depending on spatial and time unit
DATA_PATH_TRAIN = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TRAIN.parquet"
DATA_PATH_TEST = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_TEST.parquet"
DATA_PATH_VAL = INPUT / f"GOLD_{TIME_UNIT}_DEMAND_{SPATIAL_UNIT}_VAL.parquet"



MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "h3_cell", # spatial units are getting encoded
    "census_tract",
    "community_area",
    "lat",
    "lon",
    "date",
    "h3_resolution",
]

Load data and select features and target

In [26]:
# Load data
train_df = pd.read_parquet(DATA_PATH_TRAIN)
test_df = pd.read_parquet(DATA_PATH_TEST)
val_df = pd.read_parquet(DATA_PATH_VAL)

In [27]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2025-06-11 16:00:00,6,3,16,5.000000e-01,-0.866025,0.974928,-0.222521,-0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
1,2025-06-11 16:00:00,6,3,16,5.000000e-01,-0.866025,0.974928,-0.222521,-0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
2,2025-02-23 08:00:00,2,7,8,5.000000e-01,0.866025,-0.781831,0.623490,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
3,2025-02-23 08:00:00,2,7,8,5.000000e-01,0.866025,-0.781831,0.623490,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
4,2025-02-23 08:00:00,2,7,8,5.000000e-01,0.866025,-0.781831,0.623490,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71389,2026-03-27 08:00:00,3,5,8,8.660254e-01,0.500000,-0.433884,-0.900969,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
71390,2026-01-30 08:00:00,1,5,8,0.000000e+00,1.000000,-0.433884,-0.900969,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
71391,2025-07-27 08:00:00,7,7,8,1.224647e-16,-1.000000,-0.781831,0.623490,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
71392,2026-01-30 08:00:00,1,5,8,0.000000e+00,1.000000,-0.433884,-0.900969,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips


In [28]:
#print("Currently working on a sample from all the data due to runtime issues")
#train_df = train_df.sample(n=10_000, random_state=42)

In [29]:
train_df.isna().sum()

datetime_hour               0
month                       0
weekday                     0
hour                        0
month_sin                   0
                           ..
trip_total_sum              0
trip_total_mean             0
trip_total_min              0
trip_total_max              0
most_common_payment_type    0
Length: 75, dtype: int64

In [30]:
train_df["food_drink"] = train_df["food_drink"].fillna(0.0)
train_df["landmark"] = train_df["landmark"].fillna(0.0)
train_df["shop"] = train_df["shop"].fillna(0.0)
train_df["train_station"] = train_df["train_station"].fillna(0.0)

val_df["food_drink"] = val_df["food_drink"].fillna(0.0)
val_df["landmark"] = val_df["landmark"].fillna(0.0)
val_df["shop"] = val_df["shop"].fillna(0.0)
val_df["train_station"] = val_df["train_station"].fillna(0.0)

test_df["food_drink"] = test_df["food_drink"].fillna(0.0)
test_df["landmark"] = test_df["landmark"].fillna(0.0)
test_df["shop"] = test_df["shop"].fillna(0.0)
test_df["train_station"] = test_df["train_station"].fillna(0.0)

In [31]:
train_df.head()

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2025-06-11 16:00:00,6,3,16,0.5,-0.866025,0.974928,-0.222521,-0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
1,2025-06-11 16:00:00,6,3,16,0.5,-0.866025,0.974928,-0.222521,-0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
2,2025-02-23 08:00:00,2,7,8,0.5,0.866025,-0.781831,0.623490,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
3,2025-02-23 08:00:00,2,7,8,0.5,0.866025,-0.781831,0.623490,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
4,2025-02-23 08:00:00,2,7,8,0.5,0.866025,-0.781831,0.623490,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips


In [32]:
# Create X and y
def feature_cols(train_df):
    feature_cols = [
        col for col in train_df.columns
        if col not in EXCLUDE_COLS
    ]
    return feature_cols

Spatial Encoding: LatLong

In [33]:
def spherical_encode(lat, lon):
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    x = np.cos(lat_rad) * np.cos(lon_rad)
    y = np.cos(lat_rad) * np.sin(lon_rad)
    z = np.sin(lat_rad)
    return np.stack([x, y, z], axis=-1)

In [34]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2025-06-11 16:00:00,6,3,16,5.000000e-01,-0.866025,0.974928,-0.222521,-0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
1,2025-06-11 16:00:00,6,3,16,5.000000e-01,-0.866025,0.974928,-0.222521,-0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
2,2025-02-23 08:00:00,2,7,8,5.000000e-01,0.866025,-0.781831,0.623490,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
3,2025-02-23 08:00:00,2,7,8,5.000000e-01,0.866025,-0.781831,0.623490,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
4,2025-02-23 08:00:00,2,7,8,5.000000e-01,0.866025,-0.781831,0.623490,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71389,2026-03-27 08:00:00,3,5,8,8.660254e-01,0.500000,-0.433884,-0.900969,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
71390,2026-01-30 08:00:00,1,5,8,0.000000e+00,1.000000,-0.433884,-0.900969,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
71391,2025-07-27 08:00:00,7,7,8,1.224647e-16,-1.000000,-0.781831,0.623490,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips
71392,2026-01-30 08:00:00,1,5,8,0.000000e+00,1.000000,-0.433884,-0.900969,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips


In [ ]:
# encode into lat long
if (SPATIAL_ENCODING == "latlong"):
    print("Encoding: latlong and Unit: census_tract")
       
    # load census tract
    census_data = pd.read_csv(CENSUS_PATH, dtype={"CENSUS_T_1": str})
    census_data["CENSUS_T_1"] = census_data["CENSUS_T_1"].str.zfill(11)

    tract_centroids = census_data.set_index("CENSUS_T_1")[["TRACT_CE_3", "TRACT_CE_2"]]
    tract_centroids.columns = ["lat", "lon"]

    for df in (train_df, val_df, test_df):
        df["census_tract"] = df["census_tract"].astype(str).str.zfill(11)
        df["lat"] = df["census_tract"].map(tract_centroids["lat"])
        df["lon"] = df["census_tract"].map(tract_centroids["lon"])

        # sanity check, catch silent join failures early
        n_missing = df["lat"].isna().sum()
        if n_missing:
            print(f"Warning: {n_missing}/{len(df)} rows failed to match a tract centroid")
        n_missing = df["lon"].isna().sum()
        if n_missing:
            print(f"Warning: {n_missing}/{len(df)} rows failed to match a tract centroid")

Encoding: latlong and Unit: hexa


In [36]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type,lat,lon
0,2025-06-11 16:00:00,6,3,16,5.000000e-01,-0.866025,0.974928,-0.222521,-0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,41.647862,-87.620487
1,2025-06-11 16:00:00,6,3,16,5.000000e-01,-0.866025,0.974928,-0.222521,-0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,41.861837,-87.723631
2,2025-02-23 08:00:00,2,7,8,5.000000e-01,0.866025,-0.781831,0.623490,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,41.911442,-87.647322
3,2025-02-23 08:00:00,2,7,8,5.000000e-01,0.866025,-0.781831,0.623490,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,41.943521,-87.689219
4,2025-02-23 08:00:00,2,7,8,5.000000e-01,0.866025,-0.781831,0.623490,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,41.818149,-87.631380
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71389,2026-03-27 08:00:00,3,5,8,8.660254e-01,0.500000,-0.433884,-0.900969,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,42.003235,-87.802779
71390,2026-01-30 08:00:00,1,5,8,0.000000e+00,1.000000,-0.433884,-0.900969,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,41.797756,-87.639998
71391,2025-07-27 08:00:00,7,7,8,1.224647e-16,-1.000000,-0.781831,0.623490,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,41.899770,-87.596830
71392,2026-01-30 08:00:00,1,5,8,0.000000e+00,1.000000,-0.433884,-0.900969,0.866025,-0.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,41.939124,-87.718796


In [37]:
if SPATIAL_ENCODING == "latlong":
    for df in (train_df, val_df, test_df):
        result = spherical_encode(df["lat"], df["lon"])  
        df["x"], df["y"], df["z"] = result.T  

    # add x, y, z 
    feature_cols = feature_cols(train_df)

    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    X_val_grid = val_df_grid[feature_cols]

### Spatial Encoding: Spatial Embedding

In [ ]:
if (SPATIAL_ENCODING == "embedding"):
    census_data = gpd.read_file(CENSUS_PATH)
    census_data["geometry"] = gpd.GeoSeries.from_wkt(census_data["the_geom"])

    gdf = gpd.GeoDataFrame(census_data, geometry="geometry", crs="EPSG:4326")

    gdf["TRACT_FIPS"] = gdf["TRACT_FIPS"].astype(str)
    gdf = gdf.set_index("TRACT_FIPS")

    gdf = gdf.to_crs(epsg=5070)
    gdf = gdf[gdf.geometry.notnull() & gdf.geometry.is_valid]
    gdf = gdf[~gdf.index.duplicated(keep="first")]

    w = Queen.from_dataframe(gdf, use_index=True)
    G = w.to_networkx()
    print(f"Graph: {G.number_of_nodes()} tracts, {G.number_of_edges()} adjacency edges")

    node2vec = Node2Vec(
        G,
        dimensions=32,
        walk_length=20,
        num_walks=100,
        workers=4,
        p=1,
        q=1,
    )

    model = node2vec.fit(window=10, min_count=1, batch_words=4)

    embedding_dict = {node: model.wv[node] for node in G.nodes()}
    emb_df = pd.DataFrame.from_dict(embedding_dict, orient="index")

    emb_cols = [f"emb_{i}" for i in range(emb_df.shape[1])]
    emb_df.columns = emb_cols

    train_df = train_df.merge(emb_df, left_on="census_tract", right_index=True, how="left")
    val_df = val_df.merge(emb_df, left_on="census_tract", right_index=True, how="left")
    test_df = test_df.merge(emb_df, left_on="census_tract", right_index=True, how="left")

    feature_cols = feature_cols + emb_cols

    X_train = train_df[feature_cols]
    X_val = val_df[feature_cols]
    X_test = test_df[feature_cols]

    val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    X_val_grid = val_df_grid[feature_cols]


Create y

In [40]:
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]
y_val_grid = val_df_grid[TARGET_COL]

### Scale

In [41]:
X_train

,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,is_holiday,weather_station_distance_km,food_drink,landmark,...,skyc1_SCT,skyc1_BKN,skyc1_OVC,skyc1_VV,weather_station_MDW,weather_station_ORD,weather_station_IGQ,x,y,z
0,5.000000e-01,-0.866025,0.974928,-0.222521,-0.866025,-0.5,0,14.663889,0,2,...,0,0,0,0,0,0,1,0.031024,-0.746599,0.664551
1,5.000000e-01,-0.866025,0.974928,-0.222521,-0.866025,-0.5,0,8.763152,14,2,...,0,1,0,0,1,0,0,0.029581,-0.744168,0.667337
2,5.000000e-01,0.866025,-0.781831,0.623490,0.866025,-0.5,0,16.441243,174,19,...,1,0,0,0,1,0,0,0.030549,-0.743551,0.667981
3,5.000000e-01,0.866025,-0.781831,0.623490,0.866025,-0.5,0,18.280218,87,24,...,1,0,0,0,1,0,0,0.029990,-0.743199,0.668398
4,5.000000e-01,0.866025,-0.781831,0.623490,0.866025,-0.5,0,10.649357,19,7,...,1,0,0,0,1,0,0,0.030801,-0.744628,0.666769
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71389,8.660254e-01,0.500000,-0.433884,-0.900969,0.866025,-0.5,0,11.673911,19,1,...,0,1,0,0,0,1,0,0.028490,-0.742561,0.669173
71390,0.000000e+00,1.000000,-0.433884,-0.900969,0.866025,-0.5,0,9.409795,10,2,...,0,0,0,1,1,0,0,0.030698,-0.744870,0.666503
71391,1.224647e-16,-1.000000,-0.781831,0.623490,0.866025,-0.5,0,18.058670,21,27,...,1,0,0,0,1,0,0,0.031210,-0.743660,0.667830
71392,0.000000e+00,1.000000,-0.433884,-0.900969,0.866025,-0.5,0,17.252571,106,7,...,0,0,0,1,1,0,0,0.029608,-0.743266,0.668341


In [42]:
# scaling since, SVR is distance-based, so all features need to be on a similar scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_val_grid = scaler.fit_transform(X_val_grid)

### Grid Search

In [43]:
model = SVR()

In [44]:
pipe_linear = Pipeline([
    ('svm', SVR(max_iter=10000))
])

pipe_kernel = Pipeline([
    ('feature_map', Nystroem()),   # kernel set via grid
    ('svm', SVR(max_iter=10000))
])

param_grid_linear = {
    "svm__C": [0.1, 1, 10],
    "svm__epsilon": [0.1, 0.5, 1, 1.5],
}

param_grid_rbf_sigmoid = {
    "svm__C": [0.1, 1, 10],
    "svm__epsilon": [0.1, 0.5, 1, 1.5],
    "feature_map__kernel": ["rbf", "sigmoid"],
    "feature_map__gamma": [0.01, 0.1, 1],   # Nystroem doesn't accept "scale"/"auto"
    "feature_map__n_components": [100, 300, 500],
}

param_grid_poly = {
    "svm__C": [0.1, 1, 10],
    "svm__epsilon": [0.01, 0.1, 0.5, 1],
    "feature_map__kernel": ["poly"],
    "feature_map__degree": [3, 4],
    "feature_map__gamma": [0.01, 0.1, 1],
    "feature_map__n_components": [100, 300, 500],
}

grids = {}
configs = [
    ("linear", pipe_linear, param_grid_linear),
    ("rbf_sigmoid", pipe_kernel, param_grid_rbf_sigmoid),
    ("poly", pipe_kernel, param_grid_poly),
]

for name, pipe, grid in configs:
    search = GridSearchCV(
        estimator=pipe,
        param_grid=grid,
        cv=3,
        scoring="r2",
        n_jobs=-1,
        error_score="raise"
    )
    search.fit(X_val_grid, y_val_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

linear best score: 0.03814587951065459 best params: {'svm__C': 10, 'svm__epsilon': 1.5}


KeyboardInterrupt: 

In [ ]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'C': 1, 'degree': 3, 'epsilon': 1, 'gamma': 'scale', 'kernel': 'poly'}
Best CV score: 0.3961679353349734


### Train Model

In [ ]:
best_model = grid_search.best_estimator_

In [ ]:
# Train SVR 

best_model.fit(X_train, y_train)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'poly'
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",0.1
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",1
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [ ]:
# Make prediction 
y_pred = best_model.predict(X_test)

In [ ]:
y_pred

array([ 6.06623197,  1.34785082,  0.92840365, ..., -3.8927612 ,
       -0.86106653, -1.00588888])

In [ ]:
# Evaluation metrics

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 16.578284127035385
MSE: 4093.5482175240877
RMSE: 63.98084258216742
R2 Score: 0.790151368685712


In [ ]:
# save model
dump(best_model, "../models/svm/model_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_svr.joblib")
dump(grid_search, "../models/svm/grid_" + SPATIAL_UNIT +  "_" + TIME_UNIT + "_svr.joblib")

['../models/grid_community_svr.joblib']